In [16]:
import pandas as pd
import numpy as np
import os
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [17]:
# 1. Configuration
# ==========================================
DATA_DIR = 'data'
TARGET = 'Irrigation_Need'
N_FOLDS = 5
RANDOM_STATE = 42

In [18]:
print("Loading data...")
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

Loading data...


In [19]:
# Separate features and target
X = train.drop(columns=['id', TARGET])
y_raw = train[TARGET]
X_test = test.drop(columns=['id'])

In [20]:
# 3. Preprocessing
# ==========================================
print("Preprocessing data...")

# Encode Target (Low, Medium, High -> 0, 1, 2)
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
NUM_CLASSES = len(target_encoder.classes_)

Preprocessing data...


In [21]:
# Handle Categorical Features (if any exist in the dataset)
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in categorical_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')


In [22]:
# 4. Cross-Validation & Model Setup
# ==========================================
print("Setting up Cross-Validation...")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Arrays to store out-of-fold predictions and test predictions
oof_preds = np.zeros(len(train))
# For multiclass, we average the predicted probabilities across folds
test_preds_proba = np.zeros((len(test), NUM_CLASSES))

# LightGBM Parameters
lgb_params = {
    'objective': 'multiclass',
    'num_class': NUM_CLASSES,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'max_depth': 7,
    'num_leaves': 63,
    'class_weight': 'balanced', # Highly recommended for Balanced Accuracy metric
    'n_estimators': 1500,
    'random_state': RANDOM_STATE,
    'verbose': -1
}


Setting up Cross-Validation...


In [23]:
# 5. Training Loop
# ==========================================
print("Starting training...")
fold_scores =[]

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y[valid_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # Predict validation classes
    val_preds = model.predict(X_valid)
    oof_preds[valid_idx] = val_preds
    
    # Calculate fold balanced accuracy
    fold_score = balanced_accuracy_score(y_valid, val_preds)
    fold_scores.append(fold_score)
    print(f"Fold {fold+1} Balanced Accuracy: {fold_score:.5f}")
    
    # Add test probabilities
    test_preds_proba += model.predict_proba(X_test) / N_FOLDS

Starting training...
Fold 1 Balanced Accuracy: 0.96578
Fold 2 Balanced Accuracy: 0.96697
Fold 3 Balanced Accuracy: 0.96846
Fold 4 Balanced Accuracy: 0.96502
Fold 5 Balanced Accuracy: 0.96669


In [24]:
# 6. Evaluation
# ==========================================
cv_score = balanced_accuracy_score(y, oof_preds)
print("-" * 30)
print(f"Overall OOF Balanced Accuracy: {cv_score:.5f}")
print("-" * 30)

------------------------------
Overall OOF Balanced Accuracy: 0.96658
------------------------------


In [25]:
# 7. Create Submission
# ==========================================
print("Generating submission file...")

# Convert probability predictions to class indices (argmax)
final_test_preds = np.argmax(test_preds_proba, axis=1)

# Decode indices back to original labels ('Low', 'Medium', 'High')
final_test_labels = target_encoder.inverse_transform(final_test_preds)

sample_sub[TARGET] = final_test_labels
sample_sub.to_csv('submission.csv', index=False)
print("Saved to submission.csv successfully! You can now submit.")

Generating submission file...
Saved to submission.csv successfully! You can now submit.
